In [1]:
import pandas as pd
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# ============================================
# 1. Configurar chave e modelo
# ============================================

embedding_function = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")


# ============================================
# 2. Carregar e limpar o dataset
# ============================================

df = pd.read_csv('DADOS_ABERTOS_MEDICAMENTOS.csv', sep=';', encoding='iso-8859-1')

print(f"Número de registros antes da limpeza: {len(df)}")

#df_cleaned = df[df['SITUACAO_REGISTRO'].str.upper() != 'CADUCO/CANCELADO'].copy()

#print(f"Número de registros após a limpeza: {len(df_cleaned)}")

# ============================================
# 3. Converter linhas em textos semânticos
# ============================================

def row_to_text(row):
    return (
        f"TIPO DO PRODUTO: {row['TIPO_PRODUTO']}\n"
        f"NOME DO PRODUTO: {row['NOME_PRODUTO']}\n"
        f"PRINCÍPIO ATIVO: {row['PRINCIPIO_ATIVO']}\n"
        f"CLASSE TERAPÊUTICA: {row['CLASSE_TERAPEUTICA']}\n"
        f"CATEGORIA REGULATÓRIA: {row['CATEGORIA_REGULATORIA']}\n"
        f"EMPRESA DETENTORA DO REGISTRO: {row['EMPRESA_DETENTORA_REGISTRO']}\n"
        f"NÚMERO DO REGISTRO: {row['NUMERO_REGISTRO_PRODUTO']}\n"
        f"DATA DE FINALIZAÇÃO DO PROCESSO: {row['DATA_FINALIZACAO_PROCESSO']}\n"
        f"DATA DE VENCIMENTO DO REGISTRO: {row['DATA_VENCIMENTO_REGISTRO']}\n"
        f"SITUAÇÃO DO REGISTRO: {row['SITUACAO_REGISTRO']}\n"
        f"NÚMERO DO PROCESSO: {row['NUMERO_PROCESSO']}"
    )

documents = []
for _, row in df.iterrows():
    text = row_to_text(row)
    metadata = {
        "source": "DADOS_ABERTOS_MEDICAMENTOS",
        "NOME_PRODUTO": str(row["NOME_PRODUTO"]).strip(),
        "PRINCIPIO_ATIVO": str(row["PRINCIPIO_ATIVO"]).strip(),
        "CLASSE_TERAPEUTICA": str(row["CLASSE_TERAPEUTICA"]).strip(),
        "EMPRESA": str(row["EMPRESA_DETENTORA_REGISTRO"]).strip(),
    }
    documents.append(Document(page_content=text, metadata=metadata))

print(f"{len(documents)} documentos criados para o vector store.")

# ============================================
# 4. Criar vector store com Gemini embeddings
# ============================================

db = Chroma.from_documents(
    documents,
    embedding_function,
    persist_directory="../app/data/vectorstores/medicacoes_db"
)

print("✅ Vector store criado com embeddings!")


Número de registros antes da limpeza: 34555
34555 documentos criados para o vector store.
✅ Vector store criado com embeddings!


In [ ]:
# ============================================
# 5. Teste de busca
# ============================================

results = db.similarity_search("Ozempic", k=3)
print("\nResultados da busca por 'Ozempic':")
for doc in results:
    print("="*60)
    print(doc.page_content)